# Loop Control

**Topic:** `2.3` &nbsp;|&nbsp; **Module 2: Control Flow and Loops**

*break, continue and the under-used loop else: three ways a loop says how it finished, and why that is usually more expressive than a flag variable.*

> *Course:* Python Master Course &nbsp;|&nbsp; **Project:** ROBO-X (M2)

### Navigate

* [Lesson](lesson.ipynb) &nbsp;|&nbsp; [Exercises](exercises.ipynb) &nbsp;|&nbsp; [Solutions](solution.ipynb) &nbsp;|&nbsp; [Research](research.ipynb) &nbsp;|&nbsp; [Quiz](quiz.ipynb) &nbsp;|&nbsp; [Mini-project](mini_project.ipynb) &nbsp;|&nbsp; [Robotics challenge](robotics_challenge.ipynb)

Module index: [02_control_flow/README.md](../README.md)

## 1. Why This Topic Matters

A fault scanner that keeps reading after it has found the fault spends the control cycle it needed for a stop command, and a search that forgets to break reports the wrong answer with no error at all. Loop control is where a small mistake becomes a silent one.

## 2. Learning Objectives

By the end of this topic you will be able to:

1. Distinguish break, continue and return by what each one leaves.
2. Explain that break affects only the innermost loop.
3. Use the loop else to express a not-found result without a flag.
4. Recognise that return also suppresses the else clause.
5. Rewrite a flag-based search as an early-exit loop.
6. Use early exit deliberately as a performance and safety measure.

## 3. Prerequisites

This topic assumes you already have:

- Topic 2.2 for Loops and while Loops

## 4. Mental Model

**Mental model: three exits, one of which is a question.** A loop body can finish normally, skip to the next item with continue, or leave early with break. The loop else belongs to the *normal* exit only, which makes it a compact way to say "I looked at everything and found nothing".

| You write | Leaves | else runs? |
| --- | --- | --- |
| `continue` | nothing, next iteration | yes if the loop ends normally |
| `break` | the innermost loop | no |
| `return` | the whole function | no |
| body completes | nothing | yes |

## 5. Terminology

Vocabulary used consistently throughout the course.

| Term | Meaning |
| --- | --- |
| break | Exit the innermost loop immediately. |
| continue | Skip the rest of the body and start the next iteration. |
| Loop else | A clause that runs only when the loop ended without a break. |
| Early exit | Leaving a loop as soon as the answer is known. |
| Nested loop | A loop inside another loop; break affects only the inner one. |
| Flag variable | A boolean set inside a loop to record what happened. |
| Fall through | Continuing into the next statement after a loop body. |
| Search loop | A loop that stops at the first match. |
| Filter loop | A loop that skips unwanted items and keeps the rest. |
| Unreachable code | Statements after a return inside a loop that always returns. |

## 6. Conceptual Explanation

`break` leaves the innermost loop immediately. `continue` jumps to the next iteration, skipping whatever remains of the body. Both act on the *innermost* loop only, which is the single most important fact about them: in nested loops a `break` exits the inner loop and the outer one carries on.

In [ ]:
readings = [0.4, -1.0, 0.9, 0.0, 0.7]

for reading in readings:
    if reading < 0:
        continue          # skip this sample, keep scanning
    if reading > 0.95:
        break              # stop entirely
    print(f'  usable: {reading}')

print('scanned the whole list')

The third form is the one beginners never meet: an `else` clause on a loop runs **only when the loop finished without a `break`**. That turns "did I find it?" into a statement of the loop itself rather than a flag variable set afterwards.

In [ ]:
def find_dock(readings, threshold: float = 0.5):
    for index, reading in enumerate(readings):
        if reading > threshold:
            return index
    else:
        return None      # only reached if no break-equivalent fired


print(find_dock([0.1, 0.2, 0.9]))
print(find_dock([0.1, 0.2]))

> **for/else and while/else**
>
> Both forms support the else clause. It is attached to the loop, not to the if, so it is easy to misread - indent it visibly in real code and add a comment naming what 'no break' means here.

## 7. Formal Theory

The `else` clause on a loop is not the `else` of an `if`. It is a completion handler: it runs when the loop exhausts its sequence, or when a while condition first becomes false. If the loop exits through `break`, or by `return`, the `else` is skipped entirely. That makes it a direct expression of "the loop searched the whole space".

```text
for x in seq:  ...  else: <runs only if no break and no return>
```

The two exit routes through a loop body are therefore different in kind. `return` leaves the *function*, unwinding everything; `break` leaves only the innermost loop and execution continues immediately afterwards. Choosing the wrong one is a common source of code that appears to work and quietly skips the rest of the function.

In [ ]:
def first_over(values, limit):
    for value in values:
        if value > limit:
            return value        # leaves the function, skips the else
    else:
        return 'none found'


print(first_over([1.0, 5.0], 3.0))
print(first_over([1.0, 2.0], 3.0))

| Statement | Leaves | Runs the loop else? |
| --- | --- | --- |
| `break` | the innermost loop | no |
| `continue` | nothing - next iteration | yes, if the loop ends normally |
| `return` | the whole function | no |
| loop completes | nothing | yes |

## 8. Syntax

The canonical form: one `continue` for the skip case, one `break` for the search case, and an `else` for the not-found case.

In [ ]:
def first_fault(readings, limit: float):
    for index, reading in enumerate(readings):
        if reading is None:
            continue            # missing sample: not a fault
        if reading > limit:
            return index        # found it
    else:
        return None              # scanned everything, found nothing


print(first_fault([0.1, 0.8], 0.5))
print(first_fault([0.1, 0.2], 0.5))

The anti-pattern - a flag variable doing what the else clause already does:

```text
found = False
for reading in readings:
    if reading > limit:
        found = True
        break
if not found:            # the same statement, pushed outside
    report('none')

for reading in readings:
    if reading > limit:
        break
else:
    report('none')        # the loop says it directly
```

> **break only leaves one loop**
>
> In a nested loop a break exits the inner loop only. Reaching the outer loop from the inner one requires restructuring or a flag - which is usually a sign the function wants to be split.

## 9. Basic Examples

**Example 1 - filtering with continue.**

In [ ]:
readings = [0.4, None, 0.9, -0.1, 0.7]
usable = 0.0
skipped = 0
for reading in readings:
    if reading is None:
        skipped += 1
        continue
    usable += reading
print(f'sum {usable:.2f} from {len(readings) - skipped} samples')

**Example 2 - searching with break.**

In [ ]:
queue = ['a', 'b', 'shutdown', 'c', 'd']
for index, item in enumerate(queue):
    if item == 'shutdown':
        print(f'stopping at position {index}')
        break
else:
    print('queue drained without a shutdown command')

## 10. Intermediate Examples

Scanning a grid for the first obstacle is the canonical early-exit search: the moment you find it, nothing else in the grid matters.

In [ ]:
GRID = (
    (0, 0, 0, 0),
    (0, 1, 0, 0),
    (0, 0, 1, 0),
)


def first_obstacle(grid):
    for row_index, row in enumerate(grid):
        for col_index, cell in enumerate(row):
            if cell:
                return (row_index, col_index)
    return None


print('first obstacle at', first_obstacle(GRID))
print('clear grid      ', first_obstacle(((0, 0), (0, 0))))

## 11. Advanced Examples

When you genuinely need to leave *both* loops, the options are a flag, a helper function that returns, or a flattened iteration. Returning is usually the clearest, because the search becomes a value rather than a control-flow problem.

In [ ]:
def find_pair(grid, target):
    """Return the first (row, col) whose value equals target."""
    for row_index, row in enumerate(grid):
        for col_index, cell in enumerate(row):
            if cell == target:
                return row_index, col_index
    return None


print(find_pair(((1, 2), (3, 4)), 3))    # (1, 0)
print(find_pair(((1, 2), (3, 4)), 9))    # None

The same problem with a flag is correct but harder to read, because the reader must track a variable modified inside two levels of nesting. In robotics code the helper-and-return form is almost always the better trade.

## 12. Code Walkthrough

Below is a fault scanner for an underground inspection rover. It walks a telemetry batch, skips unusable samples, stops at the first real fault, and reports whether the whole batch was clean.

In [ ]:
"""scanner.py - find the first genuine fault in a telemetry batch."""
from __future__ import annotations

MAX_TEMPERATURE_C = 60.0
MAX_TILT_DEG = 20.0


def scan(batch, max_temp=MAX_TEMPERATURE_C, max_tilt=MAX_TILT_DEG) -> dict:
    """Return the first fault found, or report a clean batch.

    Samples that are None or non-numeric are skipped rather than
    treated as faults, so a dead sensor does not stop the rover.
    """
    for index, sample in enumerate(batch):
        if not isinstance(sample, dict):
            continue
        temp = sample.get('temperature_c')
        tilt = sample.get('tilt_deg')
        if temp is not None and temp > max_temp:
            return {'fault': 'overheating', 'index': index}
        if tilt is not None and tilt > max_tilt:
            return {'fault': 'tilted', 'index': index}
    else:
        return {'fault': None, 'index': None}
    return {'fault': 'unreachable', 'index': None}

The trailing `return` exists only to satisfy readers and type checkers: the `else` always returns, so it is genuinely unreachable. The `continue` for non-dict samples is the safety-relevant line - a malformed batch entry must not be reported as an overheating rover.

## 13. Line-by-Line Explanation

1. The thresholds are module constants so the safety policy is auditable without reading the loop.
2. Skipping non-dict samples with continue keeps a malformed batch entry from being reported as a hardware fault.
3. Each fault test returns immediately, so the reported index is the *first* genuine fault rather than the last one examined.
4. Using is not None rather than a truthiness test keeps a legitimate reading of zero from counting as a missing value.
5. The else clause expresses 'scanned the whole batch without finding anything', which a flag variable would have to be initialised for.
6. The final return is unreachable and exists only for readers and type checkers - a normal, honest pattern in a function that always returns.

## 14. Common Mistakes

**Mistake 1 - expecting the loop else to run after a return.**

```text
for value in values:
    if value > limit:
        return value      # leaves the function; the else never runs
else:
    return None

for value in values:
    if value > limit:
        break             # leaves the loop only; the else is skipped
else:
    report('not found')
```

**Mistake 2 - assuming break leaves both loops.**

```text
for row in grid:
    for cell in row:
        if cell == target:
            break         # leaves the inner loop only
    # the outer loop carries straight on
```

**Mistake 3 - a flag variable that is never reset.**

```text
found = False
for batch in batches:
    for sample in batch:
        if fault(sample):
            found = True
            break
    if not found:   # never reset between batches
        continue
```

**Mistake 4 - using continue where break was meant.**

```text
for reading in readings:
    if reading > limit:
        continue          # keeps scanning instead of stopping

for reading in readings:
    if reading > limit:
        break              # stop at the first offending sample
```

## 15. Debugging Techniques

When a loop-control bug is suspected, the question is always *which loop did it leave?* Instrumenting with a counter makes the exit path visible.

In [ ]:
grid = ((0, 0), (0, 0))
for row_index, row in enumerate(grid):
    for col_index, cell in enumerate(row):
        if cell == 99:
            print(f'  break exits at row {row_index} col {col_index}')
            break
    else:
        print(f'  row {row_index} had no match')
else:
    print('  whole grid scanned, no match')

A loop that never terminates when a break is expected usually has a condition that no longer holds by the time it is tested. Printing the value the break tests shows the change that was not anticipated.

In [ ]:
limit = 0.5
for reading in (0.1, 0.9, 0.2):
    if reading > limit:
        print(f'  breaking on {reading} > {limit}')
        break
    limit = 0.05   # the limit moves, so later reads may not trigger
else:
    print('  no break fired')

> **The else clause is easy to misread**
>
> Because `else` belongs to the loop rather than to the `if`, it is worth commenting: `# no break: the search completed` is often worth more than the indentation.

## 16. Best Practices

- Use `continue` to skip an unwanted item; use `break` to stop searching.
- Use the loop `else` instead of a flag variable for not-found cases.
- Remember `break` leaves only the innermost loop.
- Prefer returning from a helper over a flag set inside nested loops.
- Add a comment to any loop else, since it is easy to misread.
- Do not assume a loop else runs after a `return` from the body.
- Keep the loop body short enough that the control flow is visible.
- Test the not-found path explicitly; it is the one that is usually missing.

## 17. Performance Considerations

Early exit is a real optimisation, not just tidier code. A search that breaks on the first match examines one element; one that scans the whole sequence examines all of them. On a sensor batch with thousands of samples and an early fault, that is the difference between O(1) and O(n) in the common case.

The cost of `break` itself is nil - it is a jump. The cost of the *pattern* is what matters: a loop with a `continue` for most items can be slower than a pre-filtering comprehension, because the branch is evaluated per element. When most items are skipped, filtering before the loop is both faster and clearer; when most items are kept, the in-loop check costs almost nothing.

> **Measure the skip rate**
>
> If a loop skips most of its input, build a list of the wanted items first. If it processes most of them, leave the check where it is.

## 18. Pythonic Approaches

A flag variable versus the loop else.

In [ ]:
readings = [0.1, 0.2, 0.9]
limit = 0.5

# before: a flag, initialised and checked outside the loop
found = False
for reading in readings:
    if reading > limit:
        found = True
        break
if not found:
    print('nothing above the limit')

# after: the loop states it directly
for reading in readings:
    if reading > limit:
        print(f'found {reading}')
        break
else:
    print('nothing above the limit')

## 19. Robotics Connection

A rover must stop scanning the moment it finds a fault - continuing would waste the control cycle that a stop command needs. That is precisely what `break` is for, and the loop `else` is how it reports that the batch was clean.

In [ ]:
import sys
from pathlib import Path

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / 'shared').is_dir()), None)
if root is not None:
    sys.path.insert(0, str(root))

from shared.robo_x_sim import SimulatedRobot  # noqa: E402

robot = SimulatedRobot(name='rover-01', battery_wh=48.0)
state = robot.status()

for key, value in state.items():
    if value is None:
        continue          # a field the robot has not reported yet
    if key == 'battery_pct':
        print(f'battery {value:.1f}% - stopping the scan here')
        break
else:
    print('no battery field reported')

## 20. Engineering Example

A consumer that takes a predicate, so the same control flow serves several different search policies.

In [ ]:
def take_while_ok(items, is_ok) -> list:
    """Collect leading items while is_ok holds; stop at the first failure."""
    collected: list = []
    for item in items:
        if not is_ok(item):
            break
        collected.append(item)
    return collected

## 21. Guided Practice

1. Write a loop over a list that skips None values with continue and sums the rest.
2. Write a search that breaks on the first value above a threshold, and use the else clause to report that none was found.
3. Write a two-level nested loop with a break, and count how many times the outer body runs.
4. Rewrite a flag-variable search using the loop else and confirm the behaviour is identical.

In [ ]:
readings = [0.4, None, 0.9, -0.1, 0.7]

total = 0.0
for reading in readings:
    if reading is None:
        continue
    total += reading
print('sum:', round(total, 2))

limit = 0.6
for reading in readings:
    if reading is not None and reading > limit:
        print('first above limit:', reading)
        break
else:
    print('nothing above the limit')

runs = 0
for row in ((0, 0), (0, 0)):
    runs += 1
    for cell in row:
        if cell == 9:
            break
print('outer loop ran', runs, 'times')

## 22. Knowledge Checks

Answer these before moving on. Full answers are in [solution.ipynb](solution.ipynb).

- **Q1.** What does `break` leave?
- **Q1.** Why prefer the loop `else` to a `found` flag variable?
- **Q1.** What happens to a loop's `else` clause when the body calls `return`?
- **Q1.** Why must a rover stop scanning a telemetry batch at the first fault?

## 23. Summary

`break` leaves the innermost loop and `continue` starts the next iteration, and both act on one loop only. In nested loops that fact is the usual source of confusion: a `break` in the inner loop leaves the outer loop running, which is correct far more often than beginners expect and occasionally exactly what you want.

The third form is the one that changes how you write searches. An `else` clause on a loop runs only when the loop completed without a `break`, which is a direct statement of "I searched the whole space and found nothing". It replaces a flag variable that has to be initialised before the loop and checked after it, and it removes a whole class of bug where the flag is never reset between uses.

The subtlety to remember is that `return` also skips the loop `else`, because it leaves the function entirely. The two exits are not interchangeable, and choosing wrongly produces code that appears to work while quietly skipping the rest of the function. For a fault scanner - the robotics case this topic is built around - early exit is not only an optimisation: it is what leaves the control cycle free to issue a stop command.

## 24. Key Takeaways

- break leaves the innermost loop; continue starts the next iteration.
- The loop else runs only when no break (and no return) happened.
- Return also skips the else, because it leaves the function.
- Prefer the else clause to a flag variable for not-found cases.
- break affects one loop level; restructure or use a helper to leave two.
- Add a comment to a loop else, because it reads like an if else.
- Early exit is a real O(n) to O(1) win for search loops.
- Always test the not-found path; it is the one usually missing.

## 25. Further Exploration

- The for statement and its else clause](https://docs.python.org/3/reference/compound_stmts.html#the-for-statement)
- The while statement and its else clause](https://docs.python.org/3/reference/compound_stmts.html#the-while-statement)
- itertools - takewhile and dropwhile as loop-level control](https://docs.python.org/3/library/itertools.html)
- BREAK and CONTINUE outside a loop](https://docs.python.org/3/reference/simple_stmts.html#the-break-statement)
- PEP 709 - comprehension inlining, for future reference](https://peps.python.org/pep-0709/)